# Modul 1 - DB connection

Ostani run bazy: 03.04.2026



In [ ]:
# ============================================
# BLOK 0 — INSTALL (SAFE)
# ============================================

try:
    import paramiko
except ImportError:
    !pip install paramiko
    import paramiko

In [ ]:
# ============================================
# BLOK 1 — IMPORT DB + CONNECT (MASTER)
# ============================================

import os
import duckdb
import datetime
import paramiko
import pandas as pd

# --- KONFIG ---
VPS_HOST = "51.254.131.14"
VPS_PORT = 22
VPS_USER = "ubuntu"
VPS_PASSWORD = "ModestAmaro77#"

REMOTE_DB_PATH = "/home/ubuntu/db/forecast_db.duckdb"
LOCAL_DB_PATH = "/content/forecast_db.duckdb"

print("[INFO] START IMPORT DB")

# ============================================
# 1️⃣ ZAMKNIĘCIE STAREGO POŁĄCZENIA
# ============================================
try:
    con.close()
    print("[OK] Zamknięto stare połączenie")
except:
    print("[INFO] Brak aktywnego połączenia")

# ============================================
# 2️⃣ USUNIĘCIE LOKALNEGO PLIKU (ANTI-CACHE)
# ============================================
if os.path.exists(LOCAL_DB_PATH):
    os.remove(LOCAL_DB_PATH)
    print("[OK] Usunięto lokalną bazę")

# ============================================
# 3️⃣ DOWNLOAD Z OVH
# ============================================
print("[INFO] Pobieranie bazy z OVH...")

transport = paramiko.Transport((VPS_HOST, VPS_PORT))
transport.connect(username=VPS_USER, password=VPS_PASSWORD)

sftp = paramiko.SFTPClient.from_transport(transport)
sftp.get(REMOTE_DB_PATH, LOCAL_DB_PATH)

sftp.close()
transport.close()

print("[OK] Baza pobrana")

# ============================================
# 4️⃣ WALIDACJA PLIKU
# ============================================
if not os.path.exists(LOCAL_DB_PATH):
    raise FileNotFoundError("[ERROR] Brak pliku DB")

file_size = os.path.getsize(LOCAL_DB_PATH)
mod_time = datetime.datetime.fromtimestamp(os.path.getmtime(LOCAL_DB_PATH))

print(f"[INFO] Size: {file_size}")
print(f"[INFO] Last update: {mod_time}")

# ============================================
# 5️⃣ CONNECT DUCKDB (READ ONLY)
# ============================================
con = duckdb.connect(LOCAL_DB_PATH, read_only=True)

print("[OK] Połączono z DuckDB")

# ============================================
# 6️⃣ SANITY CHECK
# ============================================

tables = con.execute("SHOW TABLES").fetchall()
print("[INFO] Tabele:", tables)

print("\n=== VERIFY RUNS ===")

max_ts = con.execute("""
SELECT MAX(snapshot_ts) FROM runs
""").fetchone()[0]

print("MAX snapshot_ts:", max_ts)

latest_runs = con.execute("""
SELECT snapshot_ts, fixing_type
FROM runs
ORDER BY snapshot_ts DESC
LIMIT 5
""").fetchdf()

print("\nLATEST RUNS:")
print(latest_runs)

print("\n[OK] Pracujesz na AKTUALNEJ bazie")

# ============================================
# 7️⃣ TIMESTAMP SESJI
# ============================================
print(f"[INFO] Wykonano: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

[INFO] START IMPORT DB
[INFO] Brak aktywnego połączenia
[INFO] Pobieranie bazy z OVH...
[OK] Baza pobrana
[INFO] Size: 15740928
[INFO] Last update: 2026-05-14 06:15:40.834524
[OK] Połączono z DuckDB
[INFO] Tabele: [('fixing_prices',), ('forecast_pk5',), ('forecast_prices',), ('market_features',), ('market_features_raw',), ('otf_tge_ee',), ('pk5',), ('pk5_latest',), ('pk5_norm',), ('runs',), ('sdac_prices',)]

=== VERIFY RUNS ===
MAX snapshot_ts: 2026-05-14 05:18:56.074955

LATEST RUNS:
                 snapshot_ts fixing_type
0 2026-05-14 05:18:56.074955          F2
1 2026-05-14 05:12:03.988623          F1
2 2026-05-13 04:53:53.693169          F2
3 2026-05-13 04:51:55.010451          F1
4 2026-05-12 05:08:32.929526          F2

[OK] Pracujesz na AKTUALNEJ bazie
[INFO] Wykonano: 2026-05-14 06:15:41


In [ ]:
# ============================================
# BLOK 2 — LOAD DATA + FIXING CHECK + ALERT + COMPLETENESS
# ============================================

import pandas as pd
from datetime import datetime

# ============================================
# LOAD DATA
# ============================================

df = con.execute("""
SELECT *
FROM market_features_raw
ORDER BY Timestamp
""").df()

print("[OK] Dane załadowane:", df.shape)

# --- typ datetime ---
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

print("Zakres danych:")
print(df["Timestamp"].min(), "→", df["Timestamp"].max())

# ============================================
# FIXING CHECK
# ============================================

print("\n=== FIXING CHECK ===")

f1_max = df.loc[df["Fixing1"].notna(), "Timestamp"].max()
f2_max = df.loc[df["Fixing2"].notna(), "Timestamp"].max()

print("Fixing1 max delivery:", f1_max)
print("Fixing2 max delivery:", f2_max)

# ============================================
# DATA FRESHNESS CHECK
# ============================================

today = datetime.now().date()

print("\n=== DATA FRESHNESS CHECK ===")
print("Expected delivery date (D):", today)

if pd.isna(f1_max):
    print("❌ ALERT: Fixing1 BRAK DANYCH")
elif f1_max.date() < today:
    print("❌ ALERT: Fixing1 NIEAKTUALNY")
else:
    print("✅ Fixing1 OK")

if pd.isna(f2_max):
    print("❌ ALERT: Fixing2 BRAK DANYCH")
elif f2_max.date() < today:
    print("❌ ALERT: Fixing2 NIEAKTUALNY")
else:
    print("✅ Fixing2 OK")

# ============================================
# COMPLETENESS CHECK (24 GODZINY)
# ============================================

print("\n=== COMPLETENESS CHECK (H1–H24) ===")

if not pd.isna(f1_max):
    df_f1_day = df[df["Timestamp"].dt.date == f1_max.date()]
    hours_f1 = df_f1_day.loc[df_f1_day["Fixing1"].notna(), "Timestamp"].dt.hour.nunique()
    print(f"Fixing1 hours: {hours_f1}/24")

    if hours_f1 < 24:
        print("⚠️ WARNING: Fixing1 NIEPEŁNY (brak godzin)")
    else:
        print("✅ Fixing1 kompletny")
else:
    print("❌ Brak danych Fixing1 do sprawdzenia kompletności")

if not pd.isna(f2_max):
    df_f2_day = df[df["Timestamp"].dt.date == f2_max.date()]
    hours_f2 = df_f2_day.loc[df_f2_day["Fixing2"].notna(), "Timestamp"].dt.hour.nunique()
    print(f"Fixing2 hours: {hours_f2}/24")

    if hours_f2 < 24:
        print("⚠️ WARNING: Fixing2 NIEPEŁNY (brak godzin)")
    else:
        print("✅ Fixing2 kompletny")
else:
    print("❌ Brak danych Fixing2 do sprawdzenia kompletności")

# ============================================
# DATA POD WYKRES
# ============================================

df_plot = con.execute("""
SELECT
    Timestamp,
    Fixing1,
    Fixing2
FROM market_features_raw
WHERE Fixing1 IS NOT NULL
ORDER BY Timestamp
""").df()

df_plot["Timestamp"] = pd.to_datetime(df_plot["Timestamp"], errors="coerce")

print("[OK] df_plot:", df_plot.shape)

[OK] Dane załadowane: (5015, 9)
Zakres danych:
2025-11-01 01:00:00 → 2026-05-29 00:00:00

=== FIXING CHECK ===
Fixing1 max delivery: 2026-05-14 23:00:00
Fixing2 max delivery: 2026-05-14 23:00:00

=== DATA FRESHNESS CHECK ===
Expected delivery date (D): 2026-05-14
✅ Fixing1 OK
✅ Fixing2 OK

=== COMPLETENESS CHECK (H1–H24) ===
Fixing1 hours: 24/24
✅ Fixing1 kompletny
Fixing2 hours: 24/24
✅ Fixing2 kompletny
[OK] df_plot: (3838, 3)


# Modul 3 Raporty

## PLOT 1 - Fixing1 vs Fixing **2**

In [ ]:
# ============================================
# DYNAMICZNY WYKRES FORECAST (Fix1 / Fix2)
# + średnia modeli
# + checkbox Dynamiczna oś Y
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

ANALYSIS_DAYS = 7

# ------------------------------------------------
# Dostępne fixing_type
# ------------------------------------------------

available_fixings = con.execute("""
SELECT DISTINCT fixing_type
FROM runs
ORDER BY fixing_type
""").fetchdf()["fixing_type"].tolist()

fixing_dropdown = widgets.Dropdown(
    options=available_fixings,
    description="Fixing:",
    value=available_fixings[0],
    style={"description_width": "initial"}
)

scale_checkbox = widgets.Checkbox(
    value=True,
    description="Dynamiczna oś Y",
    indent=False
)

# ------------------------------------------------
# Funkcja ładowania danych
# ------------------------------------------------

def load_forecast_data(selected_fixing):

    last_run = con.execute(f"""
    SELECT run_id, forecast_start
    FROM runs
    WHERE fixing_type = '{selected_fixing}'
    ORDER BY snapshot_date DESC
    LIMIT 1
    """).fetchdf()

    if last_run.empty:
        return None, None, None

    run_id = last_run["run_id"][0]
    forecast_start = last_run["forecast_start"][0]

    df_forecast = con.execute(f"""
    SELECT *
    FROM forecast_prices
    WHERE run_id = '{run_id}'
      AND ts < '{forecast_start}'::DATE + INTERVAL '{ANALYSIS_DAYS} DAY'
    ORDER BY ts
    """).fetchdf()

    df_forecast["delivery_date"] = pd.to_datetime(df_forecast["ts"]).dt.normalize()
    df_forecast["H"] = pd.to_datetime(df_forecast["ts"]).dt.hour + 1

    return df_forecast, run_id, forecast_start


# ------------------------------------------------
# UI
# ------------------------------------------------

date_picker = widgets.DatePicker(
    description="Data dostawy:",
    style={"description_width": "initial"}
)

btn_prev = widgets.Button(description="◀ Poprzedni dzień")
btn_next = widgets.Button(description="Następny dzień ▶")

output = widgets.Output()

current_index = 0
available_deliv_dates = []
df_forecast_all = None


# ------------------------------------------------
# Funkcja rysująca wykres
# ------------------------------------------------

def plot_forecast(selected_date):

    with output:
        output.clear_output(wait=True)

        df_plot = df_forecast_all[
            df_forecast_all["delivery_date"] == selected_date
        ].copy()

        if df_plot.empty:
            print("⚠ Brak danych")
            return

        fig, ax = plt.subplots(figsize=(13,6))

        models = sorted(df_plot["model"].unique())

        hourly_values = []

        for model in models:
            sub = df_plot[df_plot["model"] == model].sort_values("H")
            avg_price = sub["price_pln_mwh"].mean()

            ax.plot(
                sub["H"],
                sub["price_pln_mwh"],
                label=f"{model} (avg: {avg_price:.0f})",
                linewidth=2
            )

            hourly_values.append(sub["price_pln_mwh"].values)

        # --- średnia modeli ---
        mean_series = np.mean(np.vstack(hourly_values), axis=0)
        mean_avg = np.mean(mean_series)

        ax.plot(
            range(1,25),
            mean_series,
            linestyle="--",
            linewidth=2.5,
            color="black",
            label=f"Średnia modeli (avg: {mean_avg:.0f})"
        )

        ax.set_xticks(range(1,25,2))  # co 2h dla lepszej czytelności

        # ------------------------------------------------
        # OŚ Y – dynamiczna lub od 0
        # ------------------------------------------------
        if scale_checkbox.value:
            y_min = df_plot["price_pln_mwh"].min()
            y_max = df_plot["price_pln_mwh"].max()
            margin = (y_max - y_min) * 0.1
            ax.set_ylim(y_min - margin, y_max + margin)
        else:
            ax.set_ylim(bottom=0)

        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(loc="upper left")

        plt.title(
            f"Forecast | {fixing_dropdown.value} | delivery: {selected_date.date()}"
        )

        plt.show()


# ------------------------------------------------
# Aktualizacja danych po zmianie fixing_type
# ------------------------------------------------

def reload_data(change=None):

    global df_forecast_all, available_deliv_dates, current_index

    df_forecast_all, run_id, forecast_start = load_forecast_data(fixing_dropdown.value)

    if df_forecast_all is None:
        print("Brak danych.")
        return

    available_deliv_dates = sorted(df_forecast_all["delivery_date"].unique())

    # START od pierwszego dnia prognozy
    current_index = 0
    date_picker.value = available_deliv_dates[current_index].date()

    update_plot()


def update_plot():
    if date_picker.value is None:
        return
    selected = pd.Timestamp(date_picker.value).normalize()
    plot_forecast(selected)


def prev_day(_):
    global current_index
    if current_index > 0:
        current_index -= 1
        date_picker.value = available_deliv_dates[current_index].date()


def next_day(_):
    global current_index
    if current_index < len(available_deliv_dates) - 1:
        current_index += 1
        date_picker.value = available_deliv_dates[current_index].date()


btn_prev.on_click(prev_day)
btn_next.on_click(next_day)

date_picker.observe(lambda change: update_plot(), names="value")
fixing_dropdown.observe(reload_data, names="value")
scale_checkbox.observe(lambda change: update_plot(), names="value")

display(widgets.HBox([
    fixing_dropdown,
    btn_prev,
    date_picker,
    btn_next,
    scale_checkbox
]))
display(output)

reload_data()

Output()

## PLOT 2 - Arbitrage: FIX1 vs FIX2

In [ ]:
# ============================================
# F1 vs F2 (forecast)
# wybór MODELU lub ŚREDNIA
# słupki = delta (F1 - F2)
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

ANALYSIS_DAYS = 7

# ------------------------------------------------
# MODELE + ŚREDNIA
# ------------------------------------------------

base_models = con.execute("""
SELECT DISTINCT model
FROM forecast_prices
ORDER BY model
""").fetchdf()["model"].tolist()

available_models = base_models + ["Średnia modeli"]

model_dropdown = widgets.Dropdown(
    options=available_models,
    description="Model:",
    value=available_models[0],
    style={"description_width": "initial"}
)

date_picker = widgets.DatePicker(
    description="Data dostawy:",
    style={"description_width": "initial"}
)

btn_prev = widgets.Button(description="◀ Poprzedni dzień")
btn_next = widgets.Button(description="Następny dzień ▶")

output = widgets.Output()

current_index = 0
available_deliv_dates = []
df_all = None


# ------------------------------------------------
# Ładowanie danych
# ------------------------------------------------

def load_data():
    global df_all

    runs_df = con.execute("""
        SELECT r.*
        FROM runs r
        JOIN (
            SELECT fixing_type, MAX(snapshot_date) AS max_snap
            FROM runs
            WHERE fixing_type IN ('F1','F2')
            GROUP BY fixing_type
        ) t
        ON r.fixing_type = t.fixing_type
        AND r.snapshot_date = t.max_snap
    """).fetchdf()

    df_list = []

    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        fixing_type = row["fixing_type"]
        forecast_start = row["forecast_start"]

        df_tmp = con.execute(f"""
            SELECT *
            FROM forecast_prices
            WHERE run_id = '{run_id}'
              AND ts < '{forecast_start}'::DATE + INTERVAL '{ANALYSIS_DAYS} DAY'
        """).fetchdf()

        if not df_tmp.empty:
            df_tmp["fixing_type"] = fixing_type
            df_list.append(df_tmp)

    df_all = pd.concat(df_list, ignore_index=True)

    df_all["delivery_date"] = pd.to_datetime(df_all["ts"]).dt.normalize()
    df_all["H"] = pd.to_datetime(df_all["ts"]).dt.hour + 1

    # wspólne daty
    fixing_groups = df_all.groupby("fixing_type")["delivery_date"].unique()
    common_dates = sorted(list(set(fixing_groups["F1"]).intersection(set(fixing_groups["F2"]))))

    df_all = df_all[df_all["delivery_date"].isin(common_dates)]


# ------------------------------------------------
# Wykres
# ------------------------------------------------

def plot_f1_vs_f2(selected_date):

    with output:
        output.clear_output(wait=True)

        if model_dropdown.value == "Średnia modeli":

            df_plot = (
                df_all[df_all["delivery_date"] == selected_date]
                .groupby(["fixing_type","H"])["price_pln_mwh"]
                .mean()
                .reset_index()
            )

        else:

            df_plot = df_all[
                (df_all["delivery_date"] == selected_date) &
                (df_all["model"] == model_dropdown.value)
            ].copy()

        f1 = df_plot[df_plot["fixing_type"] == "F1"].sort_values("H")
        f2 = df_plot[df_plot["fixing_type"] == "F2"].sort_values("H")

        if len(f1) != 24 or len(f2) != 24:
            print("⚠ Niepełne dane.")
            return

        delta = f1["price_pln_mwh"].values - f2["price_pln_mwh"].values

        f1_avg = f1["price_pln_mwh"].mean()
        f2_avg = f2["price_pln_mwh"].mean()
        d_avg  = delta.mean()

        fig, ax1 = plt.subplots(figsize=(13,6))

        ax1.plot(
            f1["H"], f1["price_pln_mwh"],
            label=f"F1 avg: {f1_avg:.0f}",
            linewidth=2
        )

        ax1.plot(
            f2["H"], f2["price_pln_mwh"],
            label=f"F2 avg: {f2_avg:.0f}",
            linewidth=2,
            linestyle="--"
        )

        ax1.set_xticks(range(1,25,2))
        ax1.grid(True, axis="y", alpha=0.3)
        ax1.legend(loc="upper left")

        ax2 = ax1.twinx()

        bars = ax2.bar(
            f1["H"],
            delta,
            alpha=0.25,
            width=0.7,
            label=f"Δ avg: {d_avg:.0f}"
        )

        for bar in bars:
            h = bar.get_height()
            ax2.text(
                bar.get_x() + bar.get_width()/2,
                h,
                f"{int(round(h))}",
                ha="center",
                va="bottom" if h >= 0 else "top",
                fontsize=8
            )

        plt.title(
            f"F1 vs F2 | {model_dropdown.value} | delivery: {selected_date.date()}"
        )

        plt.show()


# ------------------------------------------------
# Sterowanie
# ------------------------------------------------

def update_plot():
    selected = pd.Timestamp(date_picker.value).normalize()
    plot_f1_vs_f2(selected)

def prev_day(_):
    global current_index
    if current_index > 0:
        current_index -= 1
        date_picker.value = available_deliv_dates[current_index].date()

def next_day(_):
    global current_index
    if current_index < len(available_deliv_dates) - 1:
        current_index += 1
        date_picker.value = available_deliv_dates[current_index].date()

btn_prev.on_click(prev_day)
btn_next.on_click(next_day)

model_dropdown.observe(lambda change: update_plot(), names="value")
date_picker.observe(lambda change: update_plot(), names="value")


# ------------------------------------------------
# START
# ------------------------------------------------

load_data()

available_deliv_dates = sorted(df_all["delivery_date"].unique())
current_index = 0
date_picker.value = available_deliv_dates[current_index].date()

display(widgets.HBox([model_dropdown, btn_prev, date_picker, btn_next]))
display(output)

update_plot()

Output()

## PLOT 4:  German Market Update

In [ ]:
# ============================================
# MODUŁ — IMPORT DE FORECAST (OVH VPS)
# ============================================


# --- KONFIG ---
VPS_HOST = "51.254.131.14"
VPS_PORT = 22
VPS_USER = "ubuntu"
VPS_PASSWORD = "ModestAmaro77#"

REMOTE_PATH = "/home/ubuntu/data/de/DE_Price_72H_Forecast.parquet"
LOCAL_TMP_PATH = "/content/DE_Price_72H_Forecast.parquet"

# --- POŁĄCZENIE ---
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

ssh.connect(
    hostname=VPS_HOST,
    port=VPS_PORT,
    username=VPS_USER,
    password=VPS_PASSWORD
)

sftp = ssh.open_sftp()

print("[OK] Połączono z VPS")

# --- POBRANIE PLIKU ---
sftp.get(REMOTE_PATH, LOCAL_TMP_PATH)

print("[OK] Plik pobrany z VPS")

# --- ZAMKNIĘCIE POŁĄCZENIA ---
sftp.close()
ssh.close()

# --- WCZYTANIE ---
df_GER = pd.read_parquet(LOCAL_TMP_PATH)

print("[OK] Wczytano df_GER")

print("\nKolumny:")
print(df_GER.columns)

print("\nRozmiar:")
print(df_GER.shape)

print("\nPodgląd:")
display(df_GER.head())

[OK] Połączono z VPS
[OK] Plik pobrany z VPS
[OK] Wczytano df_GER

Kolumny:
Index(['datetime', 'PriceEU', 'kurs', 'PricePL', 'run_timestamp'], dtype='object')

Rozmiar:
(73, 5)

Podgląd:


,datetime,PriceEU,kurs,PricePL,run_timestamp
0,2026-05-14 07:00:00+02:00,99.04,4.28,423.8912,2026-05-14 07:10:02.658263+02:00
1,2026-05-14 08:00:00+02:00,74.85,4.28,320.3580,2026-05-14 07:10:02.658263+02:00
2,2026-05-14 09:00:00+02:00,98.99,4.28,423.6772,2026-05-14 07:10:02.658263+02:00
3,2026-05-14 10:00:00+02:00,62.75,4.28,268.5700,2026-05-14 07:10:02.658263+02:00
4,2026-05-14 11:00:00+02:00,27.69,4.28,118.5132,2026-05-14 07:10:02.658263+02:00


In [ ]:
# ============================================
# F1 vs F2 (forecast)
# + GER (PricePL)
# ============================================

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

ANALYSIS_DAYS = 7

# ------------------------------------------------
# MODELE + ŚREDNIA
# ------------------------------------------------

base_models = con.execute("""
SELECT DISTINCT model
FROM forecast_prices
ORDER BY model
""").fetchdf()["model"].tolist()

available_models = base_models + ["Średnia modeli"]

model_dropdown = widgets.Dropdown(
    options=available_models,
    description="Model:",
    value=available_models[0]
)

date_picker = widgets.DatePicker(description="Data dostawy:")
btn_prev = widgets.Button(description="◀")
btn_next = widgets.Button(description="▶")

ger_cb = widgets.Checkbox(value=False, description="GER")

output = widgets.Output()

current_index = 0
available_deliv_dates = []
df_all = None


# ------------------------------------------------
# ŁADOWANIE DANYCH
# ------------------------------------------------

def load_data():
    global df_all

    runs_df = con.execute("""
        SELECT r.*
        FROM runs r
        JOIN (
            SELECT fixing_type, MAX(snapshot_date) AS max_snap
            FROM runs
            WHERE fixing_type IN ('F1','F2')
            GROUP BY fixing_type
        ) t
        ON r.fixing_type = t.fixing_type
        AND r.snapshot_date = t.max_snap
    """).fetchdf()

    df_list = []

    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        fixing_type = row["fixing_type"]
        forecast_start = row["forecast_start"]

        df_tmp = con.execute(f"""
            SELECT *
            FROM forecast_prices
            WHERE run_id = '{run_id}'
              AND ts < '{forecast_start}'::DATE + INTERVAL '{ANALYSIS_DAYS} DAY'
        """).fetchdf()

        if not df_tmp.empty:
            df_tmp["fixing_type"] = fixing_type
            df_list.append(df_tmp)

    df_all = pd.concat(df_list, ignore_index=True)

    df_all["ts"] = pd.to_datetime(df_all["ts"])
    df_all["delivery_date"] = df_all["ts"].dt.normalize()
    df_all["H"] = df_all["ts"].dt.hour + 1

    fixing_groups = df_all.groupby("fixing_type")["delivery_date"].unique()
    common_dates = sorted(list(set(fixing_groups["F1"]).intersection(set(fixing_groups["F2"]))))

    # ✅ POPRAWIONE
    df_all = df_all[df_all["delivery_date"].isin(common_dates)]


# ------------------------------------------------
# WYKRES
# ------------------------------------------------

def plot_f1_vs_f2(selected_date):

    with output:
        clear_output(wait=True)

        if model_dropdown.value == "Średnia modeli":

            df_plot = (
                df_all[df_all["delivery_date"] == selected_date]
                .groupby(["fixing_type","H"])["price_pln_mwh"]
                .mean()
                .reset_index()
            )

        else:

            df_plot = df_all[
                (df_all["delivery_date"] == selected_date) &
                (df_all["model"] == model_dropdown.value)
            ].copy()

        f1 = df_plot[df_plot["fixing_type"] == "F1"].sort_values("H")
        f2 = df_plot[df_plot["fixing_type"] == "F2"].sort_values("H")

        if len(f1) != 24 or len(f2) != 24:
            print("⚠ Niepełne dane.")
            return

        delta = f1["price_pln_mwh"].values - f2["price_pln_mwh"].values

        fig, ax1 = plt.subplots(figsize=(13,6))

        # F1 / F2
        f1_avg = f1["price_pln_mwh"].mean()
        f2_avg = f2["price_pln_mwh"].mean()

        ax1.plot(
            f1["H"], f1["price_pln_mwh"],
            linewidth=2,
            label=f"F1 avg: {f1_avg:.0f}"
        )

        ax1.plot(
            f2["H"], f2["price_pln_mwh"],
            linewidth=2,
            linestyle="--",
            label=f"F2 avg: {f2_avg:.0f}"
        )

        # ------------------------------------------------
        # GER (PricePL)
        # ------------------------------------------------

        if ger_cb.value:

            df_ger = df_GER.copy()
            df_ger["datetime"] = pd.to_datetime(df_ger["datetime"])
            df_ger["date"] = df_ger["datetime"].dt.date

            df_ger = df_ger[df_ger["date"] == selected_date.date()]

            if not df_ger.empty:

                df_ger = df_ger.sort_values("datetime")
                df_ger["H"] = df_ger["datetime"].dt.hour + 1

                ger_avg = df_ger["PricePL"].mean()

                ax1.plot(
                    df_ger["H"],
                    df_ger["PricePL"],
                    linestyle=":",
                    linewidth=2,
                    label=f"GER avg: {ger_avg:.0f}"
                )

        ax1.legend()
        ax1.grid(True, axis="y", alpha=0.3)
        ax1.set_xticks(range(1,25,2))

        # DELTA
        ax2 = ax1.twinx()
        ax2.bar(f1["H"], delta, alpha=0.25)

        plt.title(f"{model_dropdown.value} | {selected_date.date()}")
        plt.show()


# ------------------------------------------------
# STEROWANIE
# ------------------------------------------------

def update_plot(change=None):
    selected = pd.Timestamp(date_picker.value).normalize()
    plot_f1_vs_f2(selected)

def prev_day(_):
    global current_index
    if current_index > 0:
        current_index -= 1
        date_picker.value = available_deliv_dates[current_index].date()

def next_day(_):
    global current_index
    if current_index < len(available_deliv_dates) - 1:
        current_index += 1
        date_picker.value = available_deliv_dates[current_index].date()

btn_prev.on_click(prev_day)
btn_next.on_click(next_day)

model_dropdown.observe(update_plot, names="value")
date_picker.observe(update_plot, names="value")
ger_cb.observe(update_plot, names="value")


# ------------------------------------------------
# START
# ------------------------------------------------

load_data()

available_deliv_dates = sorted(df_all["delivery_date"].unique())
current_index = 0
date_picker.value = available_deliv_dates[current_index].date()

display(widgets.HBox([model_dropdown, ger_cb, btn_prev, date_picker, btn_next]))
display(output)

update_plot()

Output()

In [ ]:
# ============================================
# GER (baseline) vs MODEL
# + delta (GER - model)
# + top/bottom 3 highlight
# ============================================

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

ANALYSIS_DAYS = 7

# ------------------------------------------------
# MODELE + ŚREDNIA
# ------------------------------------------------

base_models = con.execute("""
SELECT DISTINCT model
FROM forecast_prices
ORDER BY model
""").fetchdf()["model"].tolist()

available_models = base_models + ["Średnia modeli"]

model_dropdown = widgets.Dropdown(
    options=available_models,
    description="Model:",
    value=available_models[0]
)

date_picker = widgets.DatePicker(description="Data dostawy:")
btn_prev = widgets.Button(description="◀")
btn_next = widgets.Button(description="▶")

output = widgets.Output()

current_index = 0
available_deliv_dates = []
df_all = None


# ------------------------------------------------
# ŁADOWANIE DANYCH (FIXED)
# ------------------------------------------------

def load_data():
    global df_all

    runs_df = con.execute("""
        SELECT r.*
        FROM runs r
        JOIN (
            SELECT fixing_type, MAX(snapshot_date) AS max_snap
            FROM runs
            WHERE fixing_type IN ('F1','F2')
            GROUP BY fixing_type
        ) t
        ON r.fixing_type = t.fixing_type
        AND r.snapshot_date = t.max_snap
    """).fetchdf()

    df_list = []

    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        forecast_start = row["forecast_start"]
        fixing_type = row["fixing_type"]

        df_tmp = con.execute(f"""
            SELECT *
            FROM forecast_prices
            WHERE run_id = '{run_id}'
              AND ts < '{forecast_start}'::DATE + INTERVAL '{ANALYSIS_DAYS} DAY'
        """).fetchdf()

        if not df_tmp.empty:
            df_tmp["fixing_type"] = fixing_type   # 🔥 KLUCZOWE
            df_list.append(df_tmp)

    df_all = pd.concat(df_list, ignore_index=True)

    df_all["ts"] = pd.to_datetime(df_all["ts"])
    df_all["delivery_date"] = df_all["ts"].dt.normalize()
    df_all["H"] = df_all["ts"].dt.hour + 1


# ------------------------------------------------
# WYKRES
# ------------------------------------------------

def plot_chart(selected_date):

    with output:
        clear_output(wait=True)

        # ----------------------------------------
        # MODEL
        # ----------------------------------------

        if model_dropdown.value == "Średnia modeli":

            df_plot = (
                df_all[df_all["delivery_date"] == selected_date]
                .groupby("H")["price_pln_mwh"]
                .mean()
                .reset_index()
            )

        else:

            df_plot = df_all[
                (df_all["delivery_date"] == selected_date) &
                (df_all["model"] == model_dropdown.value) &
                (df_all["fixing_type"] == "F1")
            ][["H", "price_pln_mwh"]].copy()

        df_model = df_plot.sort_values("H")

        if len(df_model) != 24:
            print("⚠ Niepełne dane modelu.")
            return

        # ----------------------------------------
        # GER (baseline)
        # ----------------------------------------

        df_ger = df_GER.copy()
        df_ger["datetime"] = pd.to_datetime(df_ger["datetime"])
        df_ger["date"] = df_ger["datetime"].dt.date

        df_ger = df_ger[df_ger["date"] == selected_date.date()]

        if df_ger.empty:
            print("⚠ Brak danych GER.")
            return

        df_ger = df_ger.sort_values("datetime")
        df_ger["H"] = df_ger["datetime"].dt.hour + 1
        df_ger = df_ger.groupby("H")["PricePL"].mean().reset_index()

        if len(df_ger) != 24:
            print("⚠ Niepełne dane GER.")
            return

        # ----------------------------------------
        # DELTA (GER - MODEL)
        # ----------------------------------------

        delta = df_ger["PricePL"].values - df_model["price_pln_mwh"].values

        # ----------------------------------------
        # STATYSTYKI
        # ----------------------------------------

        ger_avg = df_ger["PricePL"].mean()
        model_avg = df_model["price_pln_mwh"].mean()

        # ----------------------------------------
        # TOP / BOTTOM
        # ----------------------------------------

        top_idx = delta.argsort()[-3:]
        bottom_idx = delta.argsort()[:3]

        colors = []
        for i in range(len(delta)):
            if i in top_idx:
                colors.append("green")
            elif i in bottom_idx:
                colors.append("red")
            else:
                colors.append("gray")

        # ----------------------------------------
        # PLOT
        # ----------------------------------------

        fig, ax1 = plt.subplots(figsize=(13,6))

        ax1.plot(
            df_ger["H"],
            df_ger["PricePL"],
            linestyle=":",
            linewidth=2,
            label=f"GER avg: {ger_avg:.0f}"
        )

        ax1.plot(
            df_model["H"],
            df_model["price_pln_mwh"],
            linewidth=2,
            label=f"{model_dropdown.value} avg: {model_avg:.0f}"
        )

        ax1.legend()
        ax1.grid(True, axis="y", alpha=0.3)
        ax1.set_xticks(range(1,25,2))

        # ----------------------------------------
        # DELTA BARS
        # ----------------------------------------

        ax2 = ax1.twinx()

        bars = ax2.bar(
            df_ger["H"],
            delta,
            color=colors,
            alpha=0.4
        )

        # wartości na słupkach
        for bar in bars:
            h = bar.get_height()
            ax2.text(
                bar.get_x() + bar.get_width()/2,
                h,
                f"{int(round(h))}",
                ha="center",
                va="bottom" if h >= 0 else "top",
                fontsize=8
            )

        plt.title(f"GER vs {model_dropdown.value} | {selected_date.date()}")
        plt.show()


# ------------------------------------------------
# STEROWANIE
# ------------------------------------------------

def update_plot(change=None):
    selected = pd.Timestamp(date_picker.value).normalize()
    plot_chart(selected)

def prev_day(_):
    global current_index
    if current_index > 0:
        current_index -= 1
        date_picker.value = available_deliv_dates[current_index].date()

def next_day(_):
    global current_index
    if current_index < len(available_deliv_dates) - 1:
        current_index += 1
        date_picker.value = available_deliv_dates[current_index].date()

btn_prev.on_click(prev_day)
btn_next.on_click(next_day)

model_dropdown.observe(update_plot, names="value")
date_picker.observe(update_plot, names="value")


# ------------------------------------------------
# START
# ------------------------------------------------

load_data()

available_deliv_dates = sorted(df_all["delivery_date"].unique())
current_index = 0
date_picker.value = available_deliv_dates[current_index].date()

display(widgets.HBox([model_dropdown, btn_prev, date_picker, btn_next]))
display(output)

update_plot()

Output()

# Moduł 4 Shut down DB connection

In [ ]:
# ============================================
# MODUŁ — BEZPIECZNE ZAMKNIĘCIE DUCKDB
# ============================================

import os

print("[INFO] Rozpoczynam zamykanie DuckDB...")

try:
    # 1️⃣ Flush WAL → DB (jeśli możliwe)
    try:
        con.execute("CHECKPOINT")
        print("[OK] CHECKPOINT wykonany")
    except Exception as e:
        print(f"[WARN] CHECKPOINT pominięty: {e}")

    # 2️⃣ Zamknięcie połączenia
    if con:
        con.close()
        print("[OK] Połączenie DuckDB zamknięte")

except Exception as e:
    print(f"[ERROR] Problem przy zamykaniu: {e}")

finally:
    # 3️⃣ Sync systemowy (Google Drive / FS)
    try:
        os.sync()
        print("[OK] Synchronizacja systemowa wykonana")
    except Exception as e:
        print(f"[WARN] os.sync() nieudany: {e}")

print("=== ZAMKNIĘCIE ZAKOŃCZONE ===")

[INFO] Rozpoczynam zamykanie DuckDB...
[OK] CHECKPOINT wykonany
[OK] Połączenie DuckDB zamknięte
[OK] Synchronizacja systemowa wykonana
=== ZAMKNIĘCIE ZAKOŃCZONE ===


# Prices comparission

In [ ]:
# ============================================
# RAPORT TABELARYCZNY F1 vs F2 (forecast)
# wybór DATY + MODEL / ŚREDNIA
# + przycisk EXPORT do Excel
# ============================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime
import os

ANALYSIS_DAYS = 7
EXPORT_PATH = "/content/drive/MyDrive/Forecast_Exports/"

if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)

# ------------------------------------------------
# MODELE + ŚREDNIA
# ------------------------------------------------
base_models = con.execute("""
SELECT DISTINCT model
FROM forecast_prices
ORDER BY model
""").fetchdf()["model"].tolist()

available_models = base_models + ["Średnia modeli"]

model_dropdown = widgets.Dropdown(
    options=available_models,
    description="Model:",
    value=available_models[0],
    style={"description_width": "initial"}
)

date_picker = widgets.DatePicker(
    description="Delivery date:",
    style={"description_width": "initial"}
)

btn_export = widgets.Button(
    description="Export do Excel",
    button_style="success"
)

output = widgets.Output()

# ------------------------------------------------
# Ładowanie danych (ta sama logika co wykres)
# ------------------------------------------------

def load_data():
    global df_all

    runs_df = con.execute("""
        SELECT r.*
        FROM runs r
        JOIN (
            SELECT fixing_type, MAX(snapshot_date) AS max_snap
            FROM runs
            WHERE fixing_type IN ('F1','F2')
            GROUP BY fixing_type
        ) t
        ON r.fixing_type = t.fixing_type
        AND r.snapshot_date = t.max_snap
    """).fetchdf()

    df_list = []

    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        fixing_type = row["fixing_type"]
        forecast_start = row["forecast_start"]

        df_tmp = con.execute(f"""
            SELECT *
            FROM forecast_prices
            WHERE run_id = '{run_id}'
              AND ts < '{forecast_start}'::DATE + INTERVAL '{ANALYSIS_DAYS} DAY'
        """).fetchdf()

        if not df_tmp.empty:
            df_tmp["fixing_type"] = fixing_type
            df_list.append(df_tmp)

    df_all = pd.concat(df_list, ignore_index=True)

    df_all["delivery_date"] = pd.to_datetime(df_all["ts"]).dt.normalize()
    df_all["H"] = pd.to_datetime(df_all["ts"]).dt.hour + 1

    # wspólny zakres F1 / F2
    fixing_groups = df_all.groupby("fixing_type")["delivery_date"].unique()
    common_dates = sorted(list(set(fixing_groups["F1"]).intersection(set(fixing_groups["F2"]))))

    df_all = df_all[df_all["delivery_date"].isin(common_dates)]

    return sorted(common_dates)


# ------------------------------------------------
# Budowa raportu
# ------------------------------------------------

def build_report(selected_date):

    if model_dropdown.value == "Średnia modeli":

        df_day = (
            df_all[df_all["delivery_date"] == selected_date]
            .groupby(["fixing_type","H"])["price_pln_mwh"]
            .mean()
            .reset_index()
        )

    else:

        df_day = df_all[
            (df_all["delivery_date"] == selected_date) &
            (df_all["model"] == model_dropdown.value)
        ].copy()

    f1 = df_day[df_day["fixing_type"] == "F1"][["H","price_pln_mwh"]]
    f2 = df_day[df_day["fixing_type"] == "F2"][["H","price_pln_mwh"]]

    f1 = f1.rename(columns={"price_pln_mwh":"F1_price"})
    f2 = f2.rename(columns={"price_pln_mwh":"F2_price"})

    hours_full = pd.DataFrame({"H": range(1,25)})

    df_report = hours_full.merge(f1, on="H", how="left")
    df_report = df_report.merge(f2, on="H", how="left")

    df_report["Spread"] = df_report["F1_price"] - df_report["F2_price"]

    return df_report


# ------------------------------------------------
# Wyświetlanie
# ------------------------------------------------

def show_report(change=None):

    output.clear_output()

    selected_date = date_picker.value
    if selected_date is None:
        return

    selected_ts = pd.Timestamp(selected_date).normalize()

    df_report = build_report(selected_ts)

    with output:
        display(df_report)


# ------------------------------------------------
# Export
# ------------------------------------------------

from google.colab import files

def export_excel(_):

    selected_date = date_picker.value
    if selected_date is None:
        return

    selected_ts = pd.Timestamp(selected_date).normalize()
    df_report = build_report(selected_ts)

    filename = f"F1_vs_F2_{model_dropdown.value}_{selected_ts.date()}.xlsx"
    filepath = f"/content/{filename}"

    # zapis tymczasowy
    df_report.to_excel(filepath, index=False)

    print(f"[INFO] Generuję plik: {filename}")
    files.download(filepath)


# ------------------------------------------------
# START
# ------------------------------------------------

available_dates = load_data()

date_picker.value = available_dates[0].date()

date_picker.observe(show_report, names="value")
model_dropdown.observe(show_report, names="value")
btn_export.on_click(export_excel)

display(widgets.HBox([model_dropdown, date_picker, btn_export]))
display(output)

show_report()

Output()